In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.append('/mrhome/amingk/Documents/7TPD/ActStimRL')
from utils import config 


In [2]:
# read collected data across all participants
behAll = pd.read_csv(config.RAW_BEH_ALL_FILE)


# Get unique participants with their group labels
group_counts = behAll.drop_duplicates('sub_ID')['patient'].value_counts()

# Print counts of each group
print(f"Number of Healthy Controls (HC): {group_counts.get('HC', 0)}")
print(f"Number of Parkinson's Disease (PD): {group_counts.get('PD', 0)}")


Number of Healthy Controls (HC): 27
Number of Parkinson's Disease (PD): 25


In [3]:
# Main directory of the subject
readMainDirec = '/mnt/projects/7TPD/bids/derivatives/fMRI_DA/AllBehData/'
# read collected data across all participants
behAll = pd.read_csv(config.RAW_BEH_ALL_FILE)

#  read subject registry master
Data_Status = pd.read_excel(f'{config.DATA_DIR}/AllBehData/subject_registry_Master.xlsx', sheet_name='Data_Status')

# read redcap all data
RedCAP_All_Data = pd.read_csv(f'{config.REDCAP_DIR}/all_colinical_evaluations/7TPD_RedCAP_All_Data.csv', sep=';')

# read baceline UPDRS
RedCAP_BaselineUPDRS = pd.read_csv(f'{config.REDCAP_DIR}/all_colinical_evaluations/7TPD_RedCAP_BaselineUPDRS.csv', sep=';')

# read OFF UPDRS
RedCAP_UPDRSOFF = pd.read_csv(f'{config.REDCAP_DIR}/all_colinical_evaluations/7TPD_RedCAP_UPDRSOFF.csv', sep=';')

# read ON UPDRS
RedCAP_UPDRSON = pd.read_csv(f'{config.REDCAP_DIR}/all_colinical_evaluations/7TPD_RedCAP_UPDRSON.csv', sep=';')

# read BaselineNMSS
RedCAP_BaselineNMSS = pd.read_csv(f'{config.REDCAP_DIR}/all_colinical_evaluations/7TPD_RedCAP_BaselineNMSS.csv', sep=';')

# read MOCA
RedCAP_MOCA = pd.read_csv(f'{config.REDCAP_DIR}/all_colinical_evaluations/7TPD_RedCAP_MOCA.csv', sep=';')

# read BDI
RedCAP_BDI = pd.read_csv(f'{config.REDCAP_DIR}/all_colinical_evaluations/7TPD_RedCAP_BDI.csv', sep=';')

# read LARS
RedCAP_LARS = pd.read_csv(f'{config.REDCAP_DIR}/all_colinical_evaluations/7TPD_RedCAP_LARS.csv', sep=';')

# subject ID
sub_ID = behAll['sub_ID'].unique()

In [4]:
# extract Group
group = [Data_Status[Data_Status['Unnamed: 5'].str.lower()==sub.lower()]['Unnamed: 1'].iloc[0] for sub in sub_ID]
# extract x-number
x_number = [Data_Status[Data_Status['Unnamed: 5'].str.lower()==sub.lower()]['Unnamed: 4'].iloc[0] for sub in sub_ID]

In [5]:
# Age (years)
age = [RedCAP_All_Data[RedCAP_All_Data['Record ID']==x]['Participant\'s age at inclusion'].iloc[0] for x in x_number]

In [6]:
# Sex (male/female)
sex = [RedCAP_All_Data[RedCAP_All_Data['Record ID']==x]['Sex'].iloc[0] for x in x_number]

In [7]:
# avergae time eveising duntional MRI
print(pd.to_datetime(RedCAP_All_Data['Day 3: Functional MRI - Date']).mean())
# add the aveage data to  NAN record for some participants
RedCAP_All_Data.loc[RedCAP_All_Data['Record ID'].isin(['X60049', 'X94091', 'X88722', 'X34201']), 'Day 3: Functional MRI - Date'] = '2021-05-08'


# add the aveage BDI to NAN record for some participants
RedCAP_BDI.loc[RedCAP_BDI['Record ID'].isin(['X11901', 'X42482', 'X09241', 'X62235', 'X24468']), 'Total score'] = RedCAP_BDI['Total score'].mean()


2021-05-08 10:57:23.478260992


In [8]:
# Disease Duration (years) 
disease_duration = [
    ((pd.to_datetime(RedCAP_All_Data.loc[RedCAP_All_Data['Record ID'] == x, 'Day 3: Functional MRI - Date']).iloc[0]) -
     (pd.to_datetime(RedCAP_All_Data.loc[RedCAP_All_Data['Record ID'] == x, 'Date of diagnosis']).iloc[0])
    ).days / 365.25
    for x in x_number
]

In [9]:
# Time since symptom onset (years)
time_symptomns = [
    ((pd.to_datetime(RedCAP_All_Data.loc[RedCAP_All_Data['Record ID'] == x, 'Day 3: Functional MRI - Date']).iloc[0]) -
     (pd.to_datetime(RedCAP_All_Data.loc[RedCAP_All_Data['Record ID'] == x, 'Date of symptom onset']).iloc[0])
    ).days / 365.25
    for x in x_number
]

In [10]:
# calcaulte Most affected side (left/right) from Basceline UPDRS
RedCAP_BaselineUPDRS['affected_left_right']= \
RedCAP_BaselineUPDRS['20. Tremor at rest: Hands, left']+\
RedCAP_BaselineUPDRS['20. Tremor at rest: Feet, left']+\
RedCAP_BaselineUPDRS['21. Action tremor: Left']+\
RedCAP_BaselineUPDRS['22. Rigidity: Upper extremity, left']+\
RedCAP_BaselineUPDRS['22. Rigidity: Lower extremity, left']+\
RedCAP_BaselineUPDRS['23. Finger taps: Left']+\
RedCAP_BaselineUPDRS['24. Hand grips: Left']+\
RedCAP_BaselineUPDRS['25. Hand pronate/supinate: Left']+\
RedCAP_BaselineUPDRS['26. Leg agility: Left']-\
RedCAP_BaselineUPDRS['20. Tremor at rest: Hands, right'] -\
RedCAP_BaselineUPDRS['20. Tremor at rest: Feet, right']-\
RedCAP_BaselineUPDRS['21. Action tremor: Right']-\
RedCAP_BaselineUPDRS['22. Rigidity: Upper extremity, right']-\
RedCAP_BaselineUPDRS['22. Rigidity: Lower extremity, right']-\
RedCAP_BaselineUPDRS['23. Finger taps: Right']-\
RedCAP_BaselineUPDRS['24. Hand grips: Right']-\
RedCAP_BaselineUPDRS['25. Hand pronate/supinate: Right']-\
RedCAP_BaselineUPDRS['26. Leg agility: Right']
# Add Most affected side (left/right)
most_affected_left_right_side_baselineUPDRS  = [RedCAP_BaselineUPDRS[RedCAP_BaselineUPDRS['Record ID']==x]['affected_left_right'].iloc[0] for x in x_number]

In [11]:
# calcaulte Most affected side (left/right) from OFF medication state UPDRS
RedCAP_UPDRSOFF['affected_left_right']= \
RedCAP_UPDRSOFF['20. Tremor at rest: Hands, left']+\
RedCAP_UPDRSOFF['20. Tremor at rest: Feet, left']+\
RedCAP_UPDRSOFF['21. Action tremor: Left']+\
RedCAP_UPDRSOFF['22. Rigidity: Upper extremity, left']+\
RedCAP_UPDRSOFF['22. Rigidity: Lower extremity, left']+\
RedCAP_UPDRSOFF['23. Finger taps: Left']+\
RedCAP_UPDRSOFF['24. Hand grips: Left']+\
RedCAP_UPDRSOFF['25. Hand pronate/supinate: Left']+\
RedCAP_UPDRSOFF['26. Leg agility: Left']-\
RedCAP_UPDRSOFF['20. Tremor at rest: Hands, right'] -\
RedCAP_UPDRSOFF['20. Tremor at rest: Feet, right']-\
RedCAP_UPDRSOFF['21. Action tremor: Right']-\
RedCAP_UPDRSOFF['22. Rigidity: Upper extremity, right']-\
RedCAP_UPDRSOFF['22. Rigidity: Lower extremity, right']-\
RedCAP_UPDRSOFF['23. Finger taps: Right']-\
RedCAP_UPDRSOFF['24. Hand grips: Right']-\
RedCAP_UPDRSOFF['25. Hand pronate/supinate: Right']-\
RedCAP_UPDRSOFF['26. Leg agility: Right']
# Most affected side (left/right) from OFF medication state
most_affected_left_right_side_UPDRSOFF = [RedCAP_UPDRSOFF[RedCAP_UPDRSOFF['Record ID']==x]['affected_left_right'].iloc[0] for x in x_number]

In [12]:
# calcaulte Most affected side (left/right) from ON medication state UPDRS
RedCAP_UPDRSON['affected_left_right']= \
RedCAP_UPDRSON['20. Tremor at rest: Hands, left']+\
RedCAP_UPDRSON['20. Tremor at rest: Feet, left']+\
RedCAP_UPDRSON['21. Action tremor: Left']+\
RedCAP_UPDRSON['22. Rigidity: Upper extremity, left']+\
RedCAP_UPDRSON['22. Rigidity: Lower extremity, left']+\
RedCAP_UPDRSON['23. Finger taps: Left']+\
RedCAP_UPDRSON['24. Hand grips: Left']+\
RedCAP_UPDRSON['25. Hand pronate/supinate: Left']+\
RedCAP_UPDRSON['26. Leg agility: Left']-\
RedCAP_UPDRSON['20. Tremor at rest: Hands, right'] -\
RedCAP_UPDRSON['20. Tremor at rest: Feet, right']-\
RedCAP_UPDRSON['21. Action tremor: Right']-\
RedCAP_UPDRSON['22. Rigidity: Upper extremity, right']-\
RedCAP_UPDRSON['22. Rigidity: Lower extremity, right']-\
RedCAP_UPDRSON['23. Finger taps: Right']-\
RedCAP_UPDRSON['24. Hand grips: Right']-\
RedCAP_UPDRSON['25. Hand pronate/supinate: Right']-\
RedCAP_UPDRSON['26. Leg agility: Right']
# Most affected side (left/right) from OFF medication state
most_affected_left_right_side_UPDRSON = [RedCAP_UPDRSON[RedCAP_UPDRSON['Record ID']==x]['affected_left_right'].iloc[0] for x in x_number]

In [13]:
# Systolic orthostatic blood pressure change from Basceline UPDRS
RedCAP_BaselineUPDRS['Systolic_blood_pressure_chnage'] = RedCAP_BaselineUPDRS['Blood pressure: Seated, systolic'] - RedCAP_BaselineUPDRS['Blood pressure: Standing, systolic']
systolic_blood_pressure_baselineUPDRS  = [RedCAP_BaselineUPDRS[RedCAP_BaselineUPDRS['Record ID']==x]['Systolic_blood_pressure_chnage'].iloc[0] for x in x_number]


In [14]:
# UPDRS-III(mean) from baseline UPDRS, sum of UPDRS items 1-31
RedCAP_BaselineUPDRS['total_UPDRS'] = RedCAP_BaselineUPDRS.loc[:,'1. Mentation':'31. Body bradykinesia'].sum(axis=1) 
total_BaselineUPDRS = [RedCAP_BaselineUPDRS[RedCAP_BaselineUPDRS['Record ID']==x]['total_UPDRS'].iloc[0] for x in x_number]

In [15]:
# UPDRS-III(mean) from OFF medication UPDRS, sum of UPDRS items 18-31
RedCAP_UPDRSOFF['total_UPDRS'] = RedCAP_UPDRSOFF.loc[:,'18. Speech':'31. Body bradykinesia'].sum(axis=1) 
total_UPDRSOFF = [RedCAP_UPDRSOFF[RedCAP_UPDRSOFF['Record ID']==x]['total_UPDRS'].iloc[0] for x in x_number]


In [16]:
# UPDRS-III(mean) from ON medication UPDRS, sum of UPDRS items 18-31
RedCAP_UPDRSON['total_UPDRS'] = RedCAP_UPDRSON.loc[:,'18. Speech':'31. Body bradykinesia'].sum(axis=1) 
total_UPDRSON = [RedCAP_UPDRSON[RedCAP_UPDRSON['Record ID']==x]['total_UPDRS'].iloc[0] for x in x_number]


In [17]:
# add NMSS
NMSS  = [RedCAP_BaselineNMSS[RedCAP_BaselineNMSS['Record ID']==x]['Samlet score'].iloc[0] for x in x_number]

# add MoCA
MoCA  = [RedCAP_MOCA[RedCAP_MOCA['Record ID']==x]['Total'].iloc[0] for x in x_number]

# BDI
BDI  = [RedCAP_BDI[RedCAP_BDI['Record ID']==x]['Total score'].iloc[0] for x in x_number]

# LARS
LARS  = [RedCAP_LARS[RedCAP_LARS['Record ID']==x]['Samlet score'].iloc[0] for x in x_number]

In [18]:
# add total measure to the dataframe
dict_output = {'sub_ID':sub_ID,
               'x_number':x_number,
               'group':group,
               'age':age,
               'sex':sex,
               'disease_duration':disease_duration,
               'time_symptomns':time_symptomns,
               'most_affected_left_right_side_baselineUPDRS':most_affected_left_right_side_baselineUPDRS,
               'most_affected_left_right_side_UPDRSOFF':most_affected_left_right_side_UPDRSOFF,
               'most_affected_left_right_side_UPDRSON':most_affected_left_right_side_UPDRSON,
               'systolic_blood_pressure_baselineUPDRS':systolic_blood_pressure_baselineUPDRS,
               'total_BaselineUPDRS':total_BaselineUPDRS,
               'total_UPDRSOFF':total_UPDRSOFF,
               'total_UPDRSON':total_UPDRSON,
               'NMSS':NMSS,
               'MoCA':MoCA,
               'BDI':BDI,
               'LARS':LARS}

# output clinical evaluation
df_clinical_evaluation = pd.DataFrame(dict_output)

In [19]:
# removing participnats which are noisy
withdraw_subs = ['sub-057', 'sub-076', 'sub-083', 'sub-091', 'sub-106']
for sub in withdraw_subs:
    df_clinical_evaluation = df_clinical_evaluation[df_clinical_evaluation['sub_ID']!=sub] 

In [20]:
# Some patients with PD has no UPDRS for ON and OFF but they have baseline UPDRS
df_clinical_evaluation['most_affected_left_right_side_UPDRSOFF'] = (
    df_clinical_evaluation['most_affected_left_right_side_UPDRSOFF']
    .fillna(df_clinical_evaluation['most_affected_left_right_side_baselineUPDRS']))

df_clinical_evaluation['most_affected_left_right_side_UPDRSON'] = (
    df_clinical_evaluation['most_affected_left_right_side_UPDRSON']
    .fillna(df_clinical_evaluation['most_affected_left_right_side_baselineUPDRS']))


In [21]:
# replace the NAN UPDRS ON and OFF with the meain 
meanUPDRSON_pd = round(df_clinical_evaluation.loc[df_clinical_evaluation['group']=='PD', 'total_UPDRSON'].mean())
meanUPDRSOFF_pd = round(df_clinical_evaluation.loc[df_clinical_evaluation['group']=='PD', 'total_UPDRSOFF'].mean())

df_clinical_evaluation.loc[
    (df_clinical_evaluation['group']=='PD') &
    (df_clinical_evaluation['total_UPDRSON']==0),
    'total_UPDRSON'
] = meanUPDRSON_pd

df_clinical_evaluation.loc[
    (df_clinical_evaluation['group']=='PD') &
    (df_clinical_evaluation['total_UPDRSOFF']==0),
    'total_UPDRSOFF'
] = meanUPDRSOFF_pd


In [22]:
# replace the NAN disease duration with the meain 
mean_disease_duration_pd = df_clinical_evaluation.loc[df_clinical_evaluation['group']=='PD', 'disease_duration'].mean()
mean_time_symptomns_pd = df_clinical_evaluation.loc[df_clinical_evaluation['group']=='PD', 'time_symptomns'].mean()

df_clinical_evaluation.loc[
    (df_clinical_evaluation['group']=='PD') &
    (~df_clinical_evaluation['disease_duration'].notna()),
    'disease_duration'
] = mean_disease_duration_pd
 
df_clinical_evaluation.loc[
(df_clinical_evaluation['group']=='PD') &
(~df_clinical_evaluation['time_symptomns'].notna()),
'time_symptomns'
] = mean_time_symptomns_pd


In [ ]:
# csv save colinical evaluation
df_clinical_evaluation.to_csv(f'{config.PROJECT_CLIN_EVAL_FILE}', index=False)

In [24]:
# Extract PD and HC
df_PD = df_clinical_evaluation[df_clinical_evaluation['group']=='PD']
df_HC = df_clinical_evaluation[df_clinical_evaluation['group']=='HC'][['sub_ID', 'x_number', 'group', 'age', 'sex', 'MoCA', 'BDI', 'LARS']]

In [25]:
# nummber of participants in PD
n_PD = df_PD['sub_ID'].unique().shape
print('Number of petinets with PD:  ', n_PD)
# numner for female/male
num_female = (df_PD['sex'] == 'Female').sum()
num_male = (df_PD['sex'] == 'Male').sum()
print('Female/Male in PD', num_female, '/', num_male)
# age in PD
print('Age in PD:  ', round(df_PD['age'].mean(), 2), '(', round(df_PD['age'].std(),2), ')')
# Disease duration in PD
print('Diease Duration in PD:  ', round(df_PD['disease_duration'].mean(), 2), '(', round(df_PD['disease_duration'].std(),2), ')')
# Time symtoms in PD
print('Time Symptomns in PD:  ', round(df_PD['time_symptomns'].mean(), 2), '(', round(df_PD['time_symptomns'].std(),2), ')')

# Most affected left right side UPDRSOFF in PD
print('Most affected left right side UPDRSOFF in PD:  ', np.sum(df_PD['most_affected_left_right_side_UPDRSOFF']<0), '(', np.sum(df_PD['most_affected_left_right_side_UPDRSOFF']>0), ')')

# Most affected left right side UPDRSON in PD
print('Most affected left right side UPDRSON in PD:  ', np.sum(df_PD['most_affected_left_right_side_UPDRSON']<0), '(', np.sum(df_PD['most_affected_left_right_side_UPDRSON']>0), ')')
# systolic blood pressure baseline UPDRS in PD
print('systolic blood pressure baseline UPDRS in PD:  ', round(df_PD['systolic_blood_pressure_baselineUPDRS'].mean(),2), '(', round(df_PD['systolic_blood_pressure_baselineUPDRS'].std(),2), ')')

# total UPDRS OFF in PD
print('total UPDRSOFF in PD:  ', round(df_PD['total_UPDRSOFF'].mean(),2), '(', round(df_PD['total_UPDRSOFF'].std(),2), ')')

# total UPDRS ON in PD
print('total UPDRSON in PD:  ', round(df_PD['total_UPDRSON'].mean(),2), '(', round(df_PD['total_UPDRSON'].std(),2), ')')


# NMSS in PD
print('NMSS in PD:  ', round(df_PD['NMSS'].mean(),2), '(', round(df_PD['NMSS'].std(),2), ')')

# MoCA in PD
print('MoCA in PD:  ', round(df_PD['MoCA'].mean(),2), '(', round(df_PD['MoCA'].std(),2), ')')

# BDI in PD
print('BDI in PD:  ', round(df_PD['BDI'].mean(),2), '(', round(df_PD['BDI'].std(),2), ')')

# LARS in PD
print('LARS in PD:  ', round(df_PD['LARS'].mean(),2), '(', round(df_PD['LARS'].std(),2), ')')


Number of petinets with PD:   (24,)
Female/Male in PD 10 / 14
Age in PD:   58.33 ( 9.7 )
Diease Duration in PD:   4.17 ( 2.72 )
Time Symptomns in PD:   6.39 ( 3.72 )
Most affected left right side UPDRSOFF in PD:   12 ( 12 )
Most affected left right side UPDRSON in PD:   12 ( 12 )
systolic blood pressure baseline UPDRS in PD:   14.88 ( 10.92 )
total UPDRSOFF in PD:   26.21 ( 8.03 )
total UPDRSON in PD:   19.46 ( 5.28 )
NMSS in PD:   30.83 ( 27.04 )
MoCA in PD:   27.75 ( 1.75 )
BDI in PD:   6.09 ( 4.0 )
LARS in PD:   -26.75 ( 4.67 )


In [26]:
# subjects that have no UPDRS
df_PD[~df_PD['disease_duration'].notna()][['sub_ID', 'x_number']]

,sub_ID,x_number


In [27]:
# nummber of participants in HC
n_HC = df_HC['sub_ID'].unique().shape
print('Number of petinets with HC:  ', n_HC)
# numner for female/male
num_female = (df_HC['sex'] == 'Female').sum()
num_male = (df_HC['sex'] == 'Male').sum()
print('Female/Male in HC', num_female, '/', num_male)
# age in HC
print('Age in HC:  ', round(df_HC['age'].mean(), 2), '(', round(df_HC['age'].std(),2), ')')

# MoCA in HC
print('MoCA in HC:  ', round(df_HC['MoCA'].mean(),2), '(', round(df_HC['MoCA'].std(),2), ')')

# BDI in HC
print('BDI in HC:  ', round(df_HC['BDI'].mean(),2), '(', round(df_HC['BDI'].std(),2), ')')

# LARS in HC
print('LARS in HC:  ', round(df_HC['LARS'].mean(),2), '(', round(df_HC['LARS'].std(),2), ')')


Number of petinets with HC:   (23,)
Female/Male in HC 9 / 14
Age in HC:   58.96 ( 12.04 )
MoCA in HC:   28.3 ( 1.58 )
BDI in HC:   4.28 ( 4.0 )
LARS in HC:   -28.35 ( 3.76 )


In [28]:
df_HC[~df_HC['BDI'].notna()][['sub_ID', 'x_number']]

,sub_ID,x_number
